# Exercise Recommendation LLM - Figure 3 Implementation Notebook

This notebook recreates the full system described in Lai et al. (2025):

- Domain corpus + governance + traceable data registry
- Continued pretraining (CPT)
- Supervised fine-tuning (SFT)
- Preference optimization (DPO)
- RAG grounding with citations
- Wearable-informed JITAI loop
- Safety and contraindication guardrails
- Evaluation, deployment, and compliance runbook

## How to use this notebook

1. Run cells in order.
2. Keep heavy training switches OFF until data and compute are ready.
3. Use clinician review before any real patient use.

## Scope and safety

This notebook is an engineering template for research and implementation planning. It is not medical advice and must be used with licensed clinicians and institutional governance.


In [ ]:
import os
import json
import subprocess
from pathlib import Path

# Setup flags
USE_GITHUB = True
GITHUB_REPO = "purplesquide/fitness-llm"
CLONE_DIR = "/content/fitness-llm"

REQUIRED_FILES = (
    "requirements-colab.txt",
    "pyproject.toml",
    "README.md",
)


def looks_like_project_root(path: Path) -> bool:
    return path.is_dir() and all((path / name).exists() for name in REQUIRED_FILES)


def _git(*args, cwd=None):
    result = subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)
    if result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{result.stderr}")
    return result.stdout.strip()


if USE_GITHUB:
    clone_path = Path(CLONE_DIR)
    if looks_like_project_root(clone_path):
        print(f"Repo already present at {clone_path} — pulling latest changes …")
        try:
            # Discard any local notebook mutations from previous runs so pull is clean
            _git("checkout", "HEAD", "--", "Fitness_LLM_All_In_One_Colab.ipynb", cwd=CLONE_DIR)
        except Exception:
            pass
        out = _git("pull", "--rebase", "--autostash", cwd=CLONE_DIR)
        print(f"git pull: {out or 'already up to date'}")
    else:
        print(f"Cloning {GITHUB_REPO} into {CLONE_DIR} …")
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            clone_url = f"https://{token}@github.com/{GITHUB_REPO}.git"
            print("Using GITHUB_TOKEN for private repo access")
        except Exception:
            clone_url = f"https://github.com/{GITHUB_REPO}.git"
            print("No GITHUB_TOKEN found, using public clone URL")

        _git("clone", "--depth", "1", clone_url, CLONE_DIR)

    PROJECT_ROOT = clone_path
else:
    from google.colab import drive

    drive.mount("/content/drive")
    candidates = [
        Path("/content/drive/MyDrive/fitness_llm_colab"),
        Path("/content/drive/MyDrive/fitness-llm"),
    ]
    PROJECT_ROOT = next((p for p in candidates if looks_like_project_root(p)), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Project root not found in Google Drive")

os.chdir(PROJECT_ROOT)
print(f"Working directory: {PROJECT_ROOT.resolve()}")
print(f"Has artifacts folder: {(PROJECT_ROOT / 'artifacts').exists()}")

RUNBOOK_PATH = PROJECT_ROOT / "artifacts" / "project_runbook.json"
RUNBOOK_PATH.parent.mkdir(parents=True, exist_ok=True)
RUNBOOK_PATH.write_text(json.dumps({"project_root": str(PROJECT_ROOT.resolve())}, indent=2), encoding="utf-8")
print(f"Runbook initialized: {RUNBOOK_PATH}")


In [ ]:
# Install dependencies required by all phases in this notebook.
import subprocess
import sys


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)


# Core runtime compatibility
pip("--upgrade", "--force-reinstall", "bitsandbytes>=0.45.0")
pip("--upgrade", "--force-reinstall", "trl>=0.12.0")
pip("--upgrade", "pandas>=2.2.0")

# Core project deps
pip("-r", "requirements-colab.txt")
pip("-e", ".")

# Architecture-specific packages for merged production-like pipeline
pip("--upgrade", "sentencepiece")
pip("--upgrade", "pydantic>=2.8.0")
pip("--upgrade", "qdrant-client")
pip("--upgrade", "rank-bm25")
pip("--upgrade", "datasets")
pip("--upgrade", "chromadb")
pip("--upgrade", "langchain")
pip("--upgrade", "llama-index")
pip("--upgrade", "sentence-transformers")

print("Environment setup complete.")
print("If this is your first run in a fresh Colab runtime, restart and continue from the next cell.")


In [ ]:
import os
from datetime import datetime
from huggingface_hub import login

HF_TOKEN = None

try:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN is missing. In Colab add secret HF_TOKEN, then rerun this cell."
    )

login(HF_TOKEN, add_to_git_credential=False)

PROJECT_PROFILE = {
    "project_name": "Exercise Recommendation LLM DSS",
    "timestamp_utc": datetime.utcnow().isoformat(),
    "primary_use_case": "Clinician assisted exercise recommendation",
    "target_population": "Cardiac rehab and general physical activity coaching",
    "safety_mode": "expert_in_the_loop",
    "compliance_targets": ["HIPAA", "GDPR", "FHIR", "HL7"],
}

print("HF login successful.")
print("Project profile loaded:")
print(PROJECT_PROFILE)


## Phase 1 and Phase 2 - Governance + Domain Corpus

This section implements:

- Use-case scoping and governance artifacts
- Data source registry with licensing metadata
- De-identification hooks and quality checks
- Domain dictionary build for RAG and instruction tuning

Outputs:

- artifacts/governance/project_scope.json
- artifacts/governance/risk_register.json
- artifacts/dictionaries/domain_dictionary.jsonl
- artifacts/dictionaries/user_dictionary_schema.json
- artifacts/dictionaries/user_dictionary_profiles.jsonl
- artifacts/dictionaries/instruction_dictionary.jsonl
- artifacts/datasets/fitness_training_data_dictionary_full.jsonl


In [ ]:
import json
from pathlib import Path

ROOT = Path('.')
GOV_DIR = ROOT / 'artifacts' / 'governance'
GOV_DIR.mkdir(parents=True, exist_ok=True)

project_scope = {
    "primary_use_case": "Clinician + patient exercise recommendation support",
    "initial_population": "Cardiac rehabilitation",
    "non_goals": [
        "Autonomous medical prescription without clinician sign-off",
        "Direct-to-patient unsupervised clinical exercise plans",
    ],
    "personas": [
        "clinician",
        "exercise_physiologist",
        "patient_with_clinician_oversight",
    ],
}

risk_register = {
    "regulatory_path": "Assess as SaMD candidate",
    "expert_in_loop_required": True,
    "absolute_constraints": [
        "No unsafe output delivery without rule checks",
        "No production deployment without expert evaluation",
    ],
    "team_roles": [
        "ml_engineer",
        "data_engineer",
        "clinician",
        "exercise_physiologist",
        "privacy_compliance",
        "statistician",
    ],
}

(project_scope_path := GOV_DIR / 'project_scope.json').write_text(
    json.dumps(project_scope, indent=2), encoding='utf-8'
)
(risk_register_path := GOV_DIR / 'risk_register.json').write_text(
    json.dumps(risk_register, indent=2), encoding='utf-8'
)

print(f'Wrote: {project_scope_path}')
print(f'Wrote: {risk_register_path}')


In [ ]:
# Build the full domain+user dictionaries and the fine-tune dataset.
from pathlib import Path
from fitness_llm.dictionary_builder import build_full_dictionary

result = build_full_dictionary()

print('Dictionary build complete:')
print(f'  Domain dictionary : {result.domain_jsonl}')
print(f'  User schema       : {result.user_schema_json}')
print(f'  User profiles     : {result.user_profiles_jsonl}')
print(f'  Instructions      : {result.instruction_jsonl}')
print(f'  Fine-tune dataset : {result.finetune_jsonl}')
print(f'  Summary           : {result.summary_json}')

summary = Path(result.summary_json).read_text(encoding='utf-8')
print('\nSummary preview:\n')
print(summary)


## Phase 3 and Phase 4 - Base Model Selection + Continued Pretraining (CPT)

This section sets up CPT configuration and a controlled training entrypoint.

Recommended defaults:

- Base model: mistralai/Mistral-7B-Instruct-v0.3 for Colab iteration
- CPT target corpus: domain_dictionary.jsonl body text
- Parameter-efficient method: LoRA/QLoRA
- Use replay mix if you add large domain corpora to avoid catastrophic forgetting

Keep ENABLE_CPT = False until your corpus, budget, and checkpoints are ready.


In [ ]:
import json
import os
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

# Unified operating modes
MVP_RAG_ONLY = True
ENABLE_CPT = False
ENABLE_SFT = not MVP_RAG_ONLY

# Model options: llama3 | mistral | qwen
MODEL_FAMILY = os.getenv("MODEL_FAMILY", "llama3").lower()
MODEL_CANDIDATES = {
    "llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    "mistral": "mistralai/Mistral-7B-Instruct-v0.3",
    "qwen": "Qwen/Qwen2.5-7B-Instruct",
}
BASE_MODEL_ID = os.getenv("BASE_MODEL_ID", MODEL_CANDIDATES.get(MODEL_FAMILY, MODEL_CANDIDATES["llama3"]))
MAX_SEQ_LEN = 1024

major, _ = torch.cuda.get_device_capability()
use_bf16 = major >= 8
use_fp16 = not use_bf16
dtype = torch.bfloat16 if use_bf16 else torch.float16
optim = "paged_adamw_8bit" if use_bf16 else "paged_adamw_32bit"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Base model: {BASE_MODEL_ID}")
print(f"Mode: {'MVP_RAG_ONLY' if MVP_RAG_ONLY else 'RAG_PLUS_TUNING'}")
print(f"dtype: {'bfloat16' if use_bf16 else 'float16'}, optimizer: {optim}")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = MAX_SEQ_LEN

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
    dtype=dtype,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# CPT corpus from domain dictionary
DOMAIN_DICT = "artifacts/dictionaries/domain_dictionary.jsonl"
texts = []
with open(DOMAIN_DICT, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        body = row.get("body", "").strip()
        if body:
            texts.append({"text": body})

cpt_ds = Dataset.from_list(texts)
print(f"CPT corpus rows: {len(cpt_ds)}")

if ENABLE_CPT:
    cpt_args = TrainingArguments(
        output_dir="artifacts/models/cpt-adapter",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        warmup_steps=max(1, len(cpt_ds) // 200),
        num_train_epochs=1,
        bf16=use_bf16,
        fp16=use_fp16,
        optim=optim,
        logging_steps=20,
        save_steps=200,
        report_to="none",
    )

    cpt_trainer = SFTTrainer(
        model=model,
        args=cpt_args,
        train_dataset=cpt_ds,
        processing_class=tokenizer,
        dataset_text_field="text",
        peft_config=lora,
    )

    if use_fp16:
        for _, p in cpt_trainer.model.named_parameters():
            if p.requires_grad and p.dtype == torch.bfloat16:
                p.data = p.data.to(torch.float16)

    cpt_trainer.train()
    cpt_trainer.model.save_pretrained("artifacts/models/cpt-adapter")
    tokenizer.save_pretrained("artifacts/models/cpt-adapter")
    print("CPT adapter saved to artifacts/models/cpt-adapter")
else:
    print("ENABLE_CPT is False. CPT scaffold prepared but not executed.")

## Phase 5 and Phase 6 - SFT + Preference Optimization (DPO)

This section uses the instruction dictionary and chat-formatted dataset created from the full domain and user dictionaries.

- SFT trains task behavior and output structure
- DPO adds expert preference alignment with chosen vs rejected pairs

Keep ENABLE_DPO = False until you load a real expert preference dataset.


In [ ]:
import json
from datasets import Dataset
from transformers import TrainingArguments

try:
    from trl import DPOTrainer
except Exception:
    DPOTrainer = None

ENABLE_DPO = False
SFT_DATASET_PATH = 'artifacts/datasets/fitness_training_data_dictionary_full.jsonl'

rows = []
with open(SFT_DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)
        rows.append(item)

split_idx = int(len(rows) * 0.9)
train_rows = rows[:split_idx]
eval_rows = rows[split_idx:]

# Convert chat records to plain text for TRL versions that expect text fields.
def to_text(sample):
    msgs = sample['messages']
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {'text': text}

train_ds = Dataset.from_list(train_rows).map(to_text)
eval_ds = Dataset.from_list(eval_rows).map(to_text)

print(f'SFT train: {len(train_ds)} | eval: {len(eval_ds)}')

if ENABLE_SFT:
    sft_args = TrainingArguments(
        output_dir='artifacts/models/sft-adapter',
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        lr_scheduler_type='cosine',
        warmup_steps=max(1, len(train_ds) // 200),
        num_train_epochs=3,
        bf16=use_bf16,
        fp16=use_fp16,
        optim=optim,
        evaluation_strategy='steps',
        eval_steps=100,
        logging_steps=20,
        save_steps=200,
        save_total_limit=2,
        report_to='none',
    )

    sft_trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
        dataset_text_field='text',
        peft_config=lora,
    )

    if use_fp16:
        cast_count = 0
        for _, p in sft_trainer.model.named_parameters():
            if p.requires_grad and p.dtype == torch.bfloat16:
                p.data = p.data.to(torch.float16)
                cast_count += 1
        print(f'Cast {cast_count} trainable bf16 params to fp16 for SFT.')

    sft_trainer.train()
    sft_trainer.model.save_pretrained('artifacts/models/sft-adapter')
    tokenizer.save_pretrained('artifacts/models/sft-adapter')
    print('SFT adapter saved to artifacts/models/sft-adapter')
else:
    print('ENABLE_SFT is False. SFT scaffold prepared but not executed.')

# DPO skeleton: expects JSONL with fields prompt, chosen, rejected
if ENABLE_DPO:
    if DPOTrainer is None:
        raise ImportError('DPOTrainer not available in this TRL build.')
    pref_path = 'artifacts/datasets/preference_pairs.jsonl'
    pref_rows = []
    with open(pref_path, encoding='utf-8') as f:
        for line in f:
            pref_rows.append(json.loads(line))

    pref_ds = Dataset.from_list(pref_rows)

    dpo_args = TrainingArguments(
        output_dir='artifacts/models/dpo-adapter',
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=5e-7,
        num_train_epochs=1,
        bf16=use_bf16,
        fp16=use_fp16,
        report_to='none',
    )

    dpo_trainer = DPOTrainer(
        model=sft_trainer.model if ENABLE_SFT else model,
        ref_model=None,
        args=dpo_args,
        train_dataset=pref_ds,
        processing_class=tokenizer,
    )
    dpo_trainer.train()
    dpo_trainer.model.save_pretrained('artifacts/models/dpo-adapter')
    print('DPO adapter saved to artifacts/models/dpo-adapter')
else:
    print('ENABLE_DPO is False. DPO scaffold prepared but not executed.')


In [ ]:
# Build a merged retrieval layer with FAISS+BM25 and optional ChromaDB persistence.
import json
import pickle
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

USE_CHROMA = True
CHROMA_COLLECTION = "fitness_knowledge"

DOMAIN_DICT = "artifacts/dictionaries/domain_dictionary.jsonl"
INDEX_PATH = "artifacts/rag/fitness_knowledge.index"
CHUNKS_PATH = "artifacts/rag/fitness_knowledge_chunks.pkl"
BM25_PATH = "artifacts/rag/fitness_knowledge_bm25.pkl"
EMBED_MODEL = "BAAI/bge-large-en-v1.5"

chunks = []
with open(DOMAIN_DICT, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        text = f"{row.get('title', '')}\n{row.get('body', '')}".strip()
        if text:
            chunks.append(
                {
                    "id": row.get("id"),
                    "type": row.get("type"),
                    "text": text,
                    "source": row.get("source"),
                    "tags": row.get("tags", []),
                }
            )

print(f"Domain chunks: {len(chunks)}")
embedder = SentenceTransformer(EMBED_MODEL)
texts = [c["text"] for c in chunks]
emb = embedder.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True).astype(np.float32)

# Dense local index (always built for offline reliability)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
faiss.write_index(index, INDEX_PATH)

with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(chunks, f)

# BM25 side index for lexical retrieval
bm25_tokens = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(bm25_tokens)
with open(BM25_PATH, "wb") as f:
    pickle.dump(bm25, f)

# Optional vector DB persistence for app backend integration
if USE_CHROMA:
    import chromadb

    client = chromadb.PersistentClient(path="artifacts/rag/chroma")
    try:
        client.delete_collection(CHROMA_COLLECTION)
    except Exception:
        pass

    collection = client.create_collection(name=CHROMA_COLLECTION)
    ids = [c["id"] or f"chunk_{i}" for i, c in enumerate(chunks)]
    metadatas = [{"type": c["type"], "source": c["source"]} for c in chunks]
    collection.add(ids=ids, embeddings=emb.tolist(), documents=texts, metadatas=metadatas)
    print(f"Saved Chroma collection: {CHROMA_COLLECTION}")

print(f"Saved dense index: {INDEX_PATH}")
print(f"Saved chunks: {CHUNKS_PATH}")
print(f"Saved BM25 index: {BM25_PATH}")

## Phase 7, 8, and 9 - RAG + Wearables/JITAI + Safety Guardrails

This section assembles the decision support flow from Figure 3:

1. Build patient and wearable context
2. Retrieve grounded knowledge from dense+keyword indexes
3. Generate structured recommendation draft
4. Apply contraindication and citation guardrails
5. Route low-confidence or high-risk cases to clinician review

Use this scaffold as the foundation for production orchestration and SMART-on-FHIR integration.


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════════
# FitnessCoachOrchestrator — Full Feature Merge
# Combines: multi-model generation · BGE-large hybrid RAG (FAISS+BM25) ·
#           JITAI adaptive load · exercise substitution engine ·
#           persistent user memory · Pydantic schema validation ·
#           multi-turn conversation history · safety gating · citation checks
# ═══════════════════════════════════════════════════════════════════════════════
import json
import pickle
import re
from pathlib import Path
from typing import Any, Optional
import numpy as np
import faiss
from pydantic import BaseModel, Field
from typing import List
from sentence_transformers import SentenceTransformer

# ── Pydantic response schemas ─────────────────────────────────────────────────
class ExerciseRecommendation(BaseModel):
    name: str
    sets: Optional[int] = None
    reps: Optional[str] = None
    rest_seconds: Optional[int] = None
    modifications: List[str] = Field(default_factory=list)
    source_ids: List[str] = Field(default_factory=list)

class FitnessResponse(BaseModel):
    status: str = "ok"
    plan_type: str = "general"          # strength | cardio | flexibility | recovery
    intensity_modifier: float = 1.0    # 0.0 (rest) → 1.0 (full load)
    exercises: List[ExerciseRecommendation] = Field(default_factory=list)
    coaching_note: str = ""
    escalate: bool = False
    escalation_reason: Optional[str] = None
    citations: List[str] = Field(default_factory=list)

# ── JITAI adaptive load calculator ───────────────────────────────────────────
def compute_jitai_modifier(wearable: dict) -> tuple[float, str]:
    """Return (load_modifier 0–1, coaching_note) from wearable biometrics."""
    sleep   = wearable.get("sleep_hours", 7.0)
    hrv     = wearable.get("hrv_ms", 50)
    hr      = wearable.get("hr_avg", 70)
    fatigue = wearable.get("fatigue_1_10", 5)
    steps   = wearable.get("steps", 8000)

    score, notes = 0.0, []
    if sleep < 5.5:   score -= 2; notes.append("severe sleep deficit")
    elif sleep < 6.5: score -= 1; notes.append("mild sleep deficit")
    if hrv < 30:      score -= 2; notes.append("low HRV — autonomic stress")
    elif hrv < 45:    score -= 1; notes.append("below-average HRV")
    if hr > 95:       score -= 1; notes.append("elevated resting HR")
    if fatigue >= 8:  score -= 2; notes.append("high subjective fatigue")
    elif fatigue >= 6:score -= 1; notes.append("moderate fatigue")
    if steps < 3000:  score -= 0.5; notes.append("very low activity")

    reason = ", ".join(notes) if notes else "all signals nominal"
    if score <= -4:  return 0.0,  f"Rest day required: {reason}"
    if score <= -2:  return 0.5,  f"Reduced load (50%): {reason}"
    if score <= -1:  return 0.75, f"Light session (75% load): {reason}"
    return 1.0, "Normal training load cleared."

# ── Exercise substitution engine ──────────────────────────────────────────────
_SUB_CACHE: dict[str, list[str]] = {}

def _load_sub_map(path: str = "artifacts/dictionaries/domain_dictionary.jsonl") -> dict[str, list[str]]:
    global _SUB_CACHE
    if _SUB_CACHE:
        return _SUB_CACHE
    sub_map: dict[str, list[str]] = {}
    try:
        with open(path, encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                if row.get("type") not in ("exercise", "strength", "cardio", "flexibility"):
                    continue
                title = row.get("title", "").lower().strip()
                for key in list(row.get("muscles", [])) + list(row.get("tags", [])):
                    key = str(key).lower()
                    bucket = sub_map.setdefault(key, [])
                    if title and title not in bucket:
                        bucket.append(title)
    except FileNotFoundError:
        pass
    _SUB_CACHE = sub_map
    return sub_map

def find_substitutes(exercise: str, disliked: list[str], injuries: list[str], equipment: list[str]) -> list[str]:
    sub_map   = _load_sub_map()
    norm      = exercise.lower().replace("_", " ").strip()
    blocked   = {norm} | {d.lower().replace("_", " ") for d in disliked}
    inj_keys  = [i.lower().replace("_", " ").split()[0] for i in injuries]
    candidates: set[str] = set()
    for key, bucket in sub_map.items():
        if norm in bucket or key in norm:
            candidates.update(bucket)
    filtered = [c for c in candidates if c not in blocked and not any(k in c for k in inj_keys)]
    return filtered[:3] if filtered else ["resistance band alternatives", "bodyweight alternatives"]

# ── Lazy-loaded retrieval assets ──────────────────────────────────────────────
_ASSETS: dict[str, Any] = {}

def _get_assets() -> tuple:
    if not _ASSETS:
        _ASSETS["index"]    = faiss.read_index("artifacts/rag/fitness_knowledge.index")
        _ASSETS["embedder"] = SentenceTransformer("BAAI/bge-large-en-v1.5")
        with open("artifacts/rag/fitness_knowledge_chunks.pkl", "rb") as f:
            _ASSETS["chunks"] = pickle.load(f)
        with open("artifacts/rag/fitness_knowledge_bm25.pkl", "rb") as f:
            _ASSETS["bm25"] = pickle.load(f)
    return _ASSETS["index"], _ASSETS["embedder"], _ASSETS["chunks"], _ASSETS["bm25"]

def hybrid_retrieve(query: str, top_k: int = 5) -> list[dict]:
    idx, embedder, chunks, bm25 = _get_assets()
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    d_scores, d_ids = idx.search(q_emb, top_k * 2)
    b_scores = bm25.get_scores(query.lower().split())
    b_ids = np.argsort(b_scores)[::-1][: top_k * 2]
    scored: dict[int, float] = {}
    for score, i in zip(d_scores[0], d_ids[0]):
        if i >= 0: scored[int(i)] = scored.get(int(i), 0.0) + float(score)
    for rank, i in enumerate(b_ids):
        scored[int(i)] = scored.get(int(i), 0.0) + 1.0 / (rank + 1)
    return [{**chunks[i], "score": s} for i, s in sorted(scored.items(), key=lambda x: -x[1])[:top_k]]

# ── Persistent user memory ────────────────────────────────────────────────────
_MEM_PATH = Path("artifacts/rag/user_memory_store.json")
_MEM_PATH.parent.mkdir(parents=True, exist_ok=True)
if not _MEM_PATH.exists():
    _MEM_PATH.write_text(json.dumps({"users": {}}, indent=2), encoding="utf-8")

def _load_mem()       -> dict: return json.loads(_MEM_PATH.read_text(encoding="utf-8"))
def _save_mem(m: dict) -> None: _MEM_PATH.write_text(json.dumps(m, indent=2), encoding="utf-8")

def update_user_memory(user_id: str, message: str, profile: dict) -> dict:
    mem  = _load_mem()
    slot = mem.setdefault("users", {}).setdefault(user_id, {
        "dislikes": [], "history": [], "skipped_workouts": 0, "completed_workouts": 0
    })
    lower = message.lower()
    hit = re.search(r"i (?:hate|dislike|don't like) (.+?)[\.\!\?]?$", lower)
    if hit:
        d = hit.group(1).strip()
        if d and d not in slot["dislikes"]: slot["dislikes"].append(d)
    for ex in profile.get("disliked_exercises", []):
        ex = str(ex).strip()
        if ex and ex not in slot["dislikes"]: slot["dislikes"].append(ex)
    if any(w in lower for w in ("skipped", "missed", "couldn't make it", "skip")):
        slot["skipped_workouts"] += 1
    if any(w in lower for w in ("completed", "finished", "done", "crushed")):
        slot["completed_workouts"] += 1
    slot["history"] = (slot["history"] + [message[:200]])[-10:]
    _save_mem(mem)
    return slot

# ── Safety guards ─────────────────────────────────────────────────────────────
_HARD_BLOCKS = {"unstable_angina","decompensated_heart_failure","uncontrolled_arrhythmia","acute_myocarditis"}

def has_contraindication(patient: dict) -> bool:
    return bool(set(patient.get("absolute_flags", [])) & _HARD_BLOCKS)

def validate_citations(text: str, retrieved: list[dict]) -> bool:
    cited = set(re.findall(r"\[([^\]]+)\]", text))
    if not cited: return False
    available = {r.get("id", "") for r in retrieved}
    return cited.issubset(available)

# ══════════════════════════════════════════════════════════════════════════════
class FitnessCoachOrchestrator:
    """
    Unified fitness coaching orchestrator merging all system features:
    • Pydantic-validated structured responses
    • JITAI adaptive load (wearable signals → intensity modifier)
    • Exercise substitution engine (domain dictionary lookups)
    • Hybrid RAG (FAISS dense + BM25 lexical + reciprocal rank fusion)
    • Persistent user memory (preferences, dislikes, adherence tracking)
    • Multi-turn sliding-window conversation history
    • Safety contraindication gating + citation validation
    • Multi-model generation (uses tokenizer/model from Cell 9)
    """

    def __init__(self, max_history: int = 8):
        self.max_history = max_history
        self._sessions: dict[str, list[dict]] = {}

    # ── Internal helpers ──────────────────────────────────────────────────────
    def _session(self, uid: str) -> list[dict]:
        return self._sessions.setdefault(uid, [])

    def _trim(self, uid: str):
        h = self._sessions.get(uid, [])
        if len(h) > self.max_history:
            self._sessions[uid] = h[-self.max_history:]

    def _user_ctx(self, profile: dict, wearable: dict, mem: dict, jitai: str) -> str:
        return (
            f"Level={profile.get('level','?')} Goal={profile.get('goal','?')} "
            f"Injuries={profile.get('injuries',[])} Equipment={profile.get('equipment',[])} "
            f"Dislikes={mem.get('dislikes',[])} Skipped={mem.get('skipped_workouts',0)} "
            f"HR={wearable.get('hr_avg','?')} Sleep={wearable.get('sleep_hours','?')}h "
            f"HRV={wearable.get('hrv_ms','?')}ms JITAI=[{jitai}]"
        )

    def _generate(self, messages: list[dict], max_tokens: int = 300) -> str:
        try:
            ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
            ).to(model.device)
            out = model.generate(
                ids, max_new_tokens=max_tokens, do_sample=True,
                temperature=0.72, top_p=0.9, repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
            return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
        except Exception as e:
            return f"[Generation unavailable: {e}]"

    # ── Public chat interface ─────────────────────────────────────────────────
    def chat(
        self,
        query: str,
        patient: dict,
        user_profile: dict,
        wearable_state: dict,
        user_id: str = "default_user",
    ) -> FitnessResponse:

        # 1 ▸ Absolute safety gate
        if has_contraindication(patient):
            return FitnessResponse(
                status="escalate", escalate=True,
                escalation_reason="absolute_contraindication",
                coaching_note="High-risk flags detected. Route to licensed clinician immediately.",
            )

        # 2 ▸ JITAI adaptive load
        intensity, jitai_note = compute_jitai_modifier(wearable_state)
        if intensity == 0.0:
            return FitnessResponse(
                status="ok", plan_type="recovery",
                intensity_modifier=0.0, coaching_note=jitai_note,
            )

        # 3 ▸ Persistent memory + adherence tracking
        mem = update_user_memory(user_id, query, user_profile)
        disliked  = mem.get("dislikes", [])
        injuries  = user_profile.get("injuries", [])
        equipment = user_profile.get("equipment", [])

        # 4 ▸ Build substitution hints for all disliked exercises
        subs = []
        for ex in disliked:
            alts = find_substitutes(ex, disliked, injuries, equipment)
            if alts: subs.append(f"Instead of '{ex}': {', '.join(alts)}")
        sub_block = "\n".join(subs)

        # 5 ▸ Hybrid dense+BM25 retrieval
        retrieved     = hybrid_retrieve(query, top_k=5)
        context_block = "\n\n".join(f"[{r.get('id','?')}] {r['text'][:450]}" for r in retrieved)
        user_ctx      = self._user_ctx(user_profile, wearable_state, mem, jitai_note)

        # 6 ▸ System prompt — inject all context layers
        system = (
            "You are a clinician-augmented AI fitness coach providing evidence-grounded "
            "personalised recommendations. Never diagnose. Cite sources with [id]. "
            "Respect the user's injuries, equipment, and disliked exercises. "
            f"Today's intensity modifier: {intensity:.0%}."
        )
        if sub_block:
            system += f"\n\nSubstitution suggestions (use these instead of disliked exercises):\n{sub_block}"
        if mem.get("skipped_workouts", 0) >= 2:
            system += "\n\nNote: user has recently missed sessions — emphasise gradual re-entry and motivation."

        user_msg = (
            f"User context: {user_ctx}\n\n"
            f"Retrieved evidence:\n{context_block}\n\n"
            f"User question: {query}"
        )

        # 7 ▸ Multi-turn conversation history (sliding window)
        session  = self._session(user_id)
        messages = [{"role": "system", "content": system}]
        messages.extend(session)
        messages.append({"role": "user", "content": user_msg})

        # 8 ▸ Generate
        draft = self._generate(messages, max_tokens=300)

        # 9 ▸ Update sliding session history
        session.extend([
            {"role": "user",      "content": query[:300]},
            {"role": "assistant", "content": draft[:400]},
        ])
        self._trim(user_id)

        # 10 ▸ Citation validation + structured response
        valid_cit = validate_citations(draft, retrieved)
        cited_ids = list(re.findall(r"\[([^\]]+)\]", draft))
        plan_type = "recovery" if intensity < 0.6 else ("cardio" if "cardio" in query.lower() else "general")

        return FitnessResponse(
            status="ok" if valid_cit else "needs_regeneration",
            plan_type=plan_type,
            intensity_modifier=intensity,
            coaching_note=draft,
            citations=cited_ids,
        )


# ── Instantiate global coach ──────────────────────────────────────────────────
coach = FitnessCoachOrchestrator(max_history=8)

# ── Single-turn smoke test ────────────────────────────────────────────────────
sample_patient  = {"absolute_flags": []}
sample_profile  = {
    "level": "intermediate", "goal": "hypertrophy",
    "injuries": ["left_knee_pain"],
    "equipment": ["home_dumbbells", "pullup_bar"],
    "disliked_exercises": ["barbell_back_squat"],
}
sample_wearables = {"hr_avg": 78, "sleep_hours": 6.2, "hrv_ms": 42, "steps": 5400, "fatigue_1_10": 6}

result = coach.chat(
    query="I only have dumbbells today and my knee feels irritated. What should I do?",
    patient=sample_patient,
    user_profile=sample_profile,
    wearable_state=sample_wearables,
    user_id="u_demo",
)

print(f"Status           : {result.status}")
print(f"Plan type        : {result.plan_type}")
print(f"Intensity        : {result.intensity_modifier:.0%}")
print(f"Citations        : {result.citations}")
print(f"Coaching note:\n{result.coaching_note}")


In [ ]:

# ── Multi-turn adaptive coaching demo ─────────────────────────────────────────
# Demonstrates: session memory · JITAI load changes · substitution · adherence
print("═" * 60)
print("Multi-turn conversation demo")
print("═" * 60)

turns = [
    {
        "label": "Turn 1 — normal day, preference signal",
        "query": "I hate running. What cardio can I do for fat loss today?",
        "wearables": {"hr_avg": 70, "sleep_hours": 7.5, "hrv_ms": 58, "steps": 7000, "fatigue_1_10": 3},
    },
    {
        "label": "Turn 2 — follow-up, session memory in use",
        "query": "Give me a follow-up plan for tomorrow that avoids the same movements.",
        "wearables": {"hr_avg": 72, "sleep_hours": 7.2, "hrv_ms": 55, "steps": 6800, "fatigue_1_10": 4},
    },
    {
        "label": "Turn 3 — missed session, JITAI triggers reduced load",
        "query": "I skipped yesterday's workout and feel pretty drained. What should I do today?",
        "wearables": {"hr_avg": 88, "sleep_hours": 5.0, "hrv_ms": 27, "steps": 2800, "fatigue_1_10": 8},
    },
    {
        "label": "Turn 4 — rest day triggered by signals",
        "query": "I'm exhausted. Can I still train?",
        "wearables": {"hr_avg": 95, "sleep_hours": 4.5, "hrv_ms": 20, "steps": 1500, "fatigue_1_10": 9},
    },
]

for turn in turns:
    print(f"\n{'─' * 55}")
    print(f"  {turn['label']}")
    print(f"  User: {turn['query']}")
    r = coach.chat(
        query=turn["query"],
        patient=sample_patient,
        user_profile=sample_profile,
        wearable_state=turn["wearables"],
        user_id="u_demo",
    )
    print(f"  Status: {r.status}  |  Plan: {r.plan_type}  |  Intensity: {r.intensity_modifier:.0%}")
    print(f"  Citations: {r.citations or 'none yet'}")
    print(f"  Coach:\n{r.coaching_note[:350]}{'...' if len(r.coaching_note) > 350 else ''}")

# Show final persisted memory state
import json
from pathlib import Path
mem_data = json.loads(Path("artifacts/rag/user_memory_store.json").read_text(encoding="utf-8"))
u_mem = mem_data.get("users", {}).get("u_demo", {})
print(f"\n{'═' * 60}")
print("Persisted memory for u_demo:")
print(f"  Dislikes          : {u_mem.get('dislikes', [])}")
print(f"  Skipped workouts  : {u_mem.get('skipped_workouts', 0)}")
print(f"  Completed workouts: {u_mem.get('completed_workouts', 0)}")
print(f"  History entries   : {len(u_mem.get('history', []))}")


In [ ]:

# Phase 10 — Evaluation harness using FitnessCoachOrchestrator
from dataclasses import dataclass
from typing import List


@dataclass
class EvalCase:
    case_id: str
    query: str
    expected_focus: str
    wearables: dict


benchmark_cases: List[EvalCase] = [
    EvalCase("cardiac_001",  "Post-MI 6-week progression with beta blocker",          "safety_and_hrr_logic",         {"hr_avg": 74, "sleep_hours": 7.0, "hrv_ms": 50, "steps": 5000, "fatigue_1_10": 4}),
    EvalCase("fatloss_001",  "Beginner fat-loss plan with knee pain",                  "personalization_substitution", {"hr_avg": 72, "sleep_hours": 7.5, "hrv_ms": 55, "steps": 6000, "fatigue_1_10": 3}),
    EvalCase("older_001",    "75-year-old sedentary user starting activity",            "low_impact_progression",       {"hr_avg": 70, "sleep_hours": 7.2, "hrv_ms": 48, "steps": 4000, "fatigue_1_10": 4}),
    EvalCase("strength_001", "Intermediate plateau on squat for 4 weeks",              "progressive_overload_logic",   {"hr_avg": 68, "sleep_hours": 8.0, "hrv_ms": 62, "steps": 8500, "fatigue_1_10": 2}),
    EvalCase("jitai_001",    "Low sleep and high fatigue — should JITAI reduce load?", "recovery_jitai_trigger",       {"hr_avg": 90, "sleep_hours": 5.0, "hrv_ms": 28, "steps": 2500, "fatigue_1_10": 8}),
]

eval_profile = {"level": "intermediate", "goal": "general_fitness", "injuries": [], "equipment": ["commercial_gym"], "disliked_exercises": []}


def evaluate_case(case: EvalCase) -> dict:
    r = coach.chat(
        query=case.query,
        patient={"absolute_flags": []},
        user_profile=eval_profile,
        wearable_state=case.wearables,
        user_id=f"eval_{case.case_id}",
    )
    return {
        "case_id":           case.case_id,
        "expected_focus":    case.expected_focus,
        "status":            r.status,
        "plan_type":         r.plan_type,
        "intensity":         f"{r.intensity_modifier:.0%}",
        "citations_valid":   len(r.citations) > 0,
        "draft_words":       len(r.coaching_note.split()),
        "escalated":         r.escalate,
    }


results = [evaluate_case(c) for c in benchmark_cases]
print(f"{'case_id':<16} {'focus':<32} {'status':<22} {'plan':<12} {'intens':<8} {'cit?':<6} {'words'}")
print("─" * 110)
for r in results:
    cit_flag = "✓" if r["citations_valid"] else "✗"
    print(f"{r['case_id']:<16} {r['expected_focus']:<32} {r['status']:<22} {r['plan_type']:<12} {r['intensity']:<8} {cit_flag:<6} {r['draft_words']}")


## Phase 11 and Phase 12 - Deployment, Integration, Privacy, and Regulation

Production checklist mapped to the guide:

- Serve model with vLLM or TGI
- Integrate with SMART-on-FHIR launch and FHIR CarePlan writeback
- Enforce schema + contraindication + citation validators before clinician review
- Capture accept/edit/reject decisions for monthly DPO retraining
- Implement HIPAA/GDPR data controls and audit logging
- Keep expert-in-the-loop as a hard product constraint

Clinical principle:

LLM output is a draft recommendation aid. Final plan ownership remains with licensed professionals.


In [ ]:
# Deployment and compliance runbook artifact
import json
from pathlib import Path

runbook = {
    'serving': {
        'llm_server': 'vLLM',
        'quantization': 'AWQ_INT4_or_INT8',
        'api_gateway': 'LiteLLM_or_Envoy',
    },
    'rag': {
        'vector_db': 'Qdrant',
        'retrieval': 'hybrid_dense_bm25',
        'reranker': 'bge-reranker-large',
    },
    'integration': {
        'ehr_standards': ['FHIR_R4', 'HL7_v2'],
        'auth': 'SMART_on_FHIR_OAuth2',
        'resources': ['Patient', 'Condition', 'Observation', 'MedicationStatement', 'CarePlan'],
    },
    'safety': {
        'hard_rules': True,
        'schema_validation': True,
        'citation_validation': True,
        'expert_signoff_required': True,
    },
    'privacy': {
        'encryption_at_rest': 'AES-256',
        'encryption_in_transit': 'TLS_1_3',
        'phi_redaction_in_logs': True,
        'federated_learning_candidate': True,
    },
    'evaluation_targets': {
        'rag_faithfulness_min': 0.90,
        'expert_comprehensiveness_min_pct': 80,
        'clinical_deployment_gate': 'tier1_and_tier2_passed',
    },
}

out_path = Path('artifacts/governance/deployment_runbook.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(runbook, indent=2), encoding='utf-8')

print(f'Wrote deployment runbook: {out_path}')
print(json.dumps(runbook['evaluation_targets'], indent=2))


## 12-Month Implementation Timeline and Next Actions

| Month | Focus | Deliverable |
|---|---|---|
| 1 | Scope and governance | Use case lock, risk classification, team roles |
| 2 | Domain corpus | Licensed and de-identified corpus registry |
| 3 | Baseline architecture | RAG prototype + baseline model selection |
| 4 | CPT | Domain-adapted checkpoint |
| 5 | SFT | Structured exercise recommendation model |
| 6 | DPO | Expert preference aligned checkpoint |
| 7 | RAG hardening | Citation-grounded retrieval and reranking |
| 8 | Wearables + JITAI | Streaming signals and intervention triggers |
| 9 | Safety and red-team | Contraindication and jailbreak validation report |
| 10 | Evaluation | Tier 1 automatic + Tier 2 expert pass |
| 11 | Pilot deployment | One clinical site pilot |
| 12 | Trial readiness | RCT preregistration and enrollment start |

### Final reminder

This system should be deployed as clinician-augmented decision support, not autonomous medical prescription.
